# Part 1

## 1. Setup and Configuration

In [10]:
import os
import subprocess
from dotenv import load_dotenv
from litellm import completion

load_dotenv(os.path.join("..", ".env"))

CODEBASE_DIR = os.path.abspath(os.path.join("..", "mcp-gateway-registry"))
MODEL = "anthropic/claude-sonnet-4-20250514"

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set! Add it to .env"
assert os.path.isdir(CODEBASE_DIR), f"Codebase not found at {CODEBASE_DIR}"
print(f"Codebase: {CODEBASE_DIR}")
print(f"Model: {MODEL}")
print("Setup complete!")

Codebase: /Users/satomiito/Documents/GitHub/spring-2026-a03-satomitheito/mcp-gateway-registry
Model: anthropic/claude-sonnet-4-20250514
Setup complete!


## 2. Bash Tool Execution

In [11]:
def run_bash(command: str, timeout: int = 30) -> str:
    if command.startswith("tree"):
        command = "find . -not -path '*/node_modules/*' -not -path '*/.git/*' -not -path '*/__pycache__/*' -not -path '*/.venv/*' | head -80"
    try:
        result = subprocess.run(
            command,
            shell=True,
            cwd=CODEBASE_DIR,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        output = result.stdout
        if result.stderr and result.returncode != 0:
            output += f"\n[STDERR]: {result.stderr}"
        if len(output) > 15000:
            output = output[:15000] + f"\n\n... [TRUNCATED - output was {len(output)} chars]"
        return output.strip()
    except subprocess.TimeoutExpired:
        return "[ERROR]: Command timed out"
    except Exception as e:
        return f"[ERROR]: {str(e)}"


## 3. Query Classification


In [12]:
QUERY_CATEGORIES = {
    "dependency": "Questions about project dependencies, packages, libraries",
    "structure": "Questions about project layout, entry points, file organization",
    "code_search": "Questions about specific code patterns, functions, classes",
    "architecture": "Questions about how systems work, flows, design patterns (multi-file)",
    "modification": "Questions about how to change or extend the codebase",
}

def classify_query(question: str) -> str:
    categories_str = "\n".join(f"- {k}: {v}" for k, v in QUERY_CATEGORIES.items())
    
    response = completion(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a query classifier for a code Q&A system. "
                    "Classify the user's question into exactly ONE category. "
                    f"Categories:\n{categories_str}\n\n"
                    "Respond with ONLY the category name (e.g., 'dependency'). Nothing else."
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    category = response.choices[0].message.content.strip().lower()
    if category not in QUERY_CATEGORIES:
        category = "code_search"
    return category

test_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "How does the authentication flow work?",
]
for q in test_questions:
    cat = classify_query(q)
    print(f"[{cat}] {q}")

[dependency] What Python dependencies does this project use?
[structure] What is the main entry point file for the registry service?
[architecture] How does the authentication flow work?


## 4. Bash Tool Selection


In [13]:
def get_bash_commands(category: str, question: str) -> list[str]:
    
    if category == "dependency":
        return [
            "cat pyproject.toml",
            "find . -name 'package.json' -not -path '*/node_modules/*' | head -3 | xargs -I{} sh -c 'echo \"=== {} ===\" && head -30 {}'",
            "find . -name 'requirements*.txt' -not -path '*/node_modules/*' | head -3 | xargs -I{} sh -c 'echo \"=== {} ===\" && cat {}'",
        ]
    
    elif category == "structure":
        return [
            "find . -maxdepth 2 -not -path '*/node_modules/*' -not -path '*/.git/*' -not -path '*/__pycache__/*' | sort | head -60",
            "find . -name 'main.py' -o -name 'app.py' -o -name 'index.ts' -o -name 'index.js' | grep -v node_modules | grep -v __pycache__",
            "find . -name 'Dockerfile' -o -name 'docker-compose*.yml' -o -name 'Makefile' | grep -v node_modules",
            "head -50 registry/main.py 2>/dev/null || true",
        ]
    
    elif category == "code_search":
        return _generate_search_commands(question)
    
    elif category == "architecture":
        return _generate_architecture_commands(question)
    
    elif category == "modification":
        return _generate_modification_commands(question)
    
    return ["find . -maxdepth 2 -not -path '*/.git/*' | sort | head -60"]


def _generate_search_commands(question: str) -> list[str]:
    response = completion(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You generate bash commands to search a codebase. "
                    "Given a question, output 3-4 bash commands (one per line) that would find relevant information. "
                    "Use: grep -rn (with --include filters), find, cat, head, ls. "
                    "IMPORTANT: Always pipe through 'head -40' to limit output size. "
                    "Always exclude node_modules, .git, __pycache__, .venv directories. "
                    "Do NOT use 'tree' command. Use 'find' or 'ls' instead. "
                    "Output ONLY the commands, one per line. No explanations or markdown."
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    commands = [
        line.strip().lstrip("$ ").lstrip("`").rstrip("`")
        for line in response.choices[0].message.content.strip().split("\n")
        if line.strip() and not line.strip().startswith("#") and not line.strip().startswith("```")
    ]
    return commands[:4]


def _generate_architecture_commands(question: str) -> list[str]:
    response = completion(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You generate bash commands to explore a codebase's architecture. "
                    "The codebase is an MCP Gateway Registry with: "
                    "registry/ (FastAPI backend), auth_server/ (OAuth service), cli/ (TypeScript), "
                    "frontend/ (React), docs/ (markdown docs), tests/. "
                    "Given a question, output 4-5 bash commands to gather relevant context from code AND docs. "
                    "Include: grep -rn for patterns, head for key files, find for structure. "
                    "IMPORTANT: Always pipe through 'head -50' to limit output size. "
                    "Exclude node_modules, .git, __pycache__, .venv. "
                    "Do NOT use 'tree' command. "
                    "Output ONLY commands, one per line. No explanations or markdown."
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    commands = [
        line.strip().lstrip("$ ").lstrip("`").rstrip("`")
        for line in response.choices[0].message.content.strip().split("\n")
        if line.strip() and not line.strip().startswith("#") and not line.strip().startswith("```")
    ]
    return commands[:5]


def _generate_modification_commands(question: str) -> list[str]:
    response = completion(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You generate bash commands to understand how to extend/modify a codebase. "
                    "The codebase is an MCP Gateway Registry with: "
                    "registry/ (FastAPI backend), auth_server/ (OAuth service with providers/), cli/ (TypeScript), "
                    "frontend/ (React), docs/ (markdown docs), tests/. "
                    "Given a question about modifying the codebase, output 4-6 commands that: "
                    "1) Find existing similar implementations (as patterns to follow), "
                    "2) Find interfaces/base classes, "
                    "3) Find config files that need updating, "
                    "4) Search docs for relevant guides. "
                    "IMPORTANT: Always pipe through 'head -50' to limit output. "
                    "Exclude node_modules, .git, __pycache__, .venv. "
                    "Do NOT use 'tree' command. "
                    "Output ONLY commands, one per line. No explanations or markdown."
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0,
    )
    commands = [
        line.strip().lstrip("$ ").lstrip("`").rstrip("`")
        for line in response.choices[0].message.content.strip().split("\n")
        if line.strip() and not line.strip().startswith("#") and not line.strip().startswith("```")
    ]
    return commands[:6]

## 5. Context Retrieval Pipeline


In [14]:
def retrieve_context(category: str, question: str) -> str:
    commands = get_bash_commands(category, question)
    
    context_parts = []
    print(f"\n Category: {category}")
    print(f"Executing {len(commands)} bash commands:")
    
    for cmd in commands:
        output = run_bash(cmd)
        if output and output != "[ERROR]: Command timed out":
            context_parts.append(f"--- Command: {cmd} ---\n{output}")
    
    return "\n\n".join(context_parts)

## 6. Answer Generation


In [15]:
def generate_answer(question: str, context: str) -> str:
    response = completion(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a code Q&A assistant. Answer the user's question based on the "
                    "provided codebase context (output from bash commands). "
                    "Always reference specific files and paths in your answer. "
                    "If the context doesn't contain enough information, say what's missing. "
                    "Be thorough but concise."
                ),
            },
            {
                "role": "user",
                "content": f"## Question\n{question}\n\n## Codebase Context\n{context}",
            },
        ],
        temperature=0,
    )
    return response.choices[0].message.content

## 7. Full Pipeline


In [16]:
def code_qa(question: str) -> dict:
    print(f"\n Question: {question}")

    category = classify_query(question)
    
    context = retrieve_context(category, question)
    
    answer = generate_answer(question, context)
    
    print(f"\n Answer:\n{answer}")
    print("\n" + "=" * 80)
    
    return {
        "question": question,
        "category": category,
        "answer": answer,
    }

## 8. Run Test Questions

In [17]:
test_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)",
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?",
]

results = []
for q in test_questions:
    result = code_qa(q)
    results.append(result)


 Question: What Python dependencies does this project use?

 Category: dependency
Executing 3 bash commands:

 Answer:
Based on the codebase context, this project uses dependencies across multiple languages:

## Python Dependencies (from `pyproject.toml`)

The main Python project has **42 core dependencies** including:

### Web Framework & API
- `fastapi>=0.115.12` - Web framework
- `uvicorn[standard]>=0.34.2` - ASGI server
- `httpx>=0.27.0` - HTTP client
- `aiohttp>=3.8.0` - Async HTTP client/server
- `websockets>=15.0.1` - WebSocket support

### MCP & AI/ML
- `mcp>=1.9.3` - Model Context Protocol
- `langchain-mcp-adapters>=0.0.11` - LangChain MCP integration
- `langgraph>=0.4.3` - LangGraph framework
- `langchain-anthropic>=0.3.17` - Anthropic integration
- `langchain-aws>=0.2.23` - AWS integration
- `sentence-transformers>=3.0.0` - Sentence embeddings
- `faiss-cpu>=1.7.4` - Vector similarity search
- `torch>=1.6.0` - PyTorch
- `scikit-learn>=1.3.0` - Machine learning
- `huggingface

In [18]:
output_path = os.path.join("..", "part1_results.txt")

with open(output_path, "w") as f:
    f.write("Part 1: Code Q&A System with Bash Tools - Results\n")
    
    for i, r in enumerate(results, 1):
        f.write(f"Question {i}: {r['question']}\n")
        f.write(f"Category: {r['category']}\n")
        f.write(f"{r['answer']}\n")

print(f"Results saved to {os.path.abspath(output_path)}")

Results saved to /Users/satomiito/Documents/GitHub/spring-2026-a03-satomitheito/part1_results.txt
